In [18]:
#!import C:\Users\musharm\source\repos\performance_dynamic\src\benchmarks\gc\GC.Infrastructure\Notebooks\BenchmarkAnalysis_Fundamentals.ipynb

  Determining projects to restore...
  All projects are up-to-date for restore.
  GC.Analysis.API -> C:\Users\musharm\source\repos\performance_dynamic\artifacts\bin\GC.Analysis.API\Release\net8.0\GC.Analysis.API.dll

Build succeeded.
    0 Warning(s)
    0 Error(s)

Time Elapsed 00:00:01.57


Restore sources https://pkgs.dev.azure.com/dnceng/public/_packaging/dotnet-public/nuget/v3/index.json Installed Packages Microsoft.Data.Analysis, 0.19.1 Newtonsoft.Json, 13.0.3 XPlot.Plotly, 4.0.6 XPlot.Plotly.Interactive, 4.0.7 YamlDotnet, 15.1.2

In [19]:
var fragmentationFixPath = @"C:\Users\musharm\source\repos\performance\src\benchmarks\gc\GC.Infrastructure\Configurations\ASPNetBenchmarks\FragmentationFix_Data";
var dm = DataManager.CreateAspNetData(ML(fragmentationFixPath));

In [20]:
static Metric<TraceGC> excessFragmentation = new Metric<TraceGC>((d => {
    Dictionary<CondemnedReasonGroup, int> condemnReasonsPerGC = new Dictionary<CondemnedReasonGroup, int>();
    d.GetCondemnedReasons(condemnReasonsPerGC);
    return condemnReasonsPerGC.ContainsKey(CondemnedReasonGroup.Fragmented_Ephemeral) ? 1 : 0; 
}), "IsExcessEphFrag", "#");
static Aggregation sumAggregation = new Aggregation((d => d.Sum()), "Sum", "#");
static Metric<IterationData> iterationExcessFragmentation = Metrics.Promote(excessFragmentation, sumAggregation);
static Metric<BenchmarkData> TotalExcessFragmentation = Metrics.Promote(iterationExcessFragmentation, Aggregation.Average);

### Existence of Issue in Fix

In [26]:
CompareInfo info = new CompareInfo("datas_nofix", "datas_fix");
TableBenchmarks(dm, ML( TotalExcessFragmentation ), configFilter: new Filter( ML ( "datas_fix", "datas_nofix" )), compareInfo: info);


Per-benchmark behavior / Average of Sum of IsExcessEphFrag

|                                 |             |           |            |
|                                 |             |           | Comparison |
|                  Benchmark Name | datas_nofix | datas_fix |          % |
| --------------------------------|-------------|-----------|----------- |
|          ConnectionCloseHttpSys |       3.250 |     0.000 |   -100.000 |
|                      FortunesEf |       2.500 |     0.000 |   -100.000 |
|              FortunesPlatformEF |       1.750 |     0.000 |   -100.000 |
|                       JsonHttps |       0.250 |     0.000 |   -100.000 |
|                         JsonMvc |       0.500 |     0.000 |   -100.000 |
|         MultipleQueriesPlatform |       2.250 |     0.000 |   -100.000 |
|                    PlaintextMvc |   1,104.000 |     0.000 |   -100.000 |
| PlaintextWithParametersNoFilter |     437.250 |     0.000 |   -100.000 |
|             SingleQueryPlatform |    

### Server Fix vs. DATAS Fix

In [22]:
TableBenchmarks(dm, ML( Metrics.B.AverageMaxHeapSize, Metrics.B.AverageRequestPerMSec, Metrics.B.AverageP50Latency), configFilter: new Filter( ML ( "server_fix", "datas_fix" )),
 compareInfo: new CompareInfo("server_fix", "datas_fix"));


Per-benchmark behavior

|                                 |            |   Average | Average of |            |           |            |            |   Average | Average of |
|                                 | Average of |    of Max |   Max heap |            |           | Average of | Average of |        of |    Latency |
|                                 |   Max heap | heap size |    size /  | Average of |   Average |     RPS /  |    Latency |   Latency |    50th /  |
|                                 |     size / |         / | Comparison |      RPS / |  of RPS / | Comparison |     50th / |    50th / | Comparison |
|                  Benchmark Name | server_fix | datas_fix |          % | server_fix | datas_fix |          % | server_fix | datas_fix |          % |
| --------------------------------|------------|-----------|------------|------------|-----------|------------|------------|-----------|----------- |
|          ConnectionCloseHttpSys |    352.582 |    20.692 |    -94.131 |  

### Server No Fix vs. DATAS No Fix

In [23]:
TableBenchmarks(dm, ML( Metrics.B.AverageMaxHeapSize, Metrics.B.AverageRequestPerMSec, Metrics.B.AverageP50Latency), configFilter: new Filter( ML ( "server_nofix", "datas_nofix" )),
 compareInfo: new CompareInfo("server_nofix", "datas_nofix"));


Per-benchmark behavior

|                                 |              |             | Average of |              |             |            |              |             | Average of |
|                                 |   Average of |  Average of |   Max heap |              |             | Average of |   Average of |  Average of |    Latency |
|                                 |     Max heap |    Max heap |    size /  |   Average of |  Average of |     RPS /  | Latency 50th |     Latency |    50th /  |
|                                 |       size / |      size / | Comparison |        RPS / |       RPS / | Comparison |            / |      50th / | Comparison |
|                  Benchmark Name | server_nofix | datas_nofix |          % | server_nofix | datas_nofix |          % | server_nofix | datas_nofix |          % |
| --------------------------------|--------------|-------------|------------|--------------|-------------|------------|--------------|-------------|----------- |
|  

In [24]:
//  ChartGCData(dm, ML( excessFragmentation )); //, dataFilter: d => d.DynamicEvents().SizeAdaptationTuning?.TcpToConsider != 0  );

In [25]:
//ChartGCData(dm, ML( Metrics.G.NumHeaps, Metrics.G.Generation, Metrics.G.MedianThroughputCostPercent) , dataFilter: d => d.DynamicEvents().SizeAdaptationTuning?.TcpToConsider != 0  );